In [1]:
!pip install -q -U bitsandbytes
import torch, transformers, bitsandbytes
print("GPU:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__, "| bitsandbytes:", bitsandbytes.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.5 MB/s eta 0:00:00
GPU: Tesla T4
transformers: 5.16.1 | bitsandbytes: 0.50.2


In [2]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM = ("You sanitize Romanized Bangla (Banglish) prompts. Return only JSON with keys: "
          "sanitized_prompt, pii, preserved_entities, uncertainties, routing.")
EXAMPLE = "kalk3 bkash e 5oo tk send korsi trans id 9X87K, taka pay nai. 1987654321 eta check koren"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"          # needed when generating several prompts at once
prompt_text = tokenizer.apply_chat_template(
    [{"role": "system", "content": SYSTEM}, {"role": "user", "content": EXAMPLE}],
    add_generation_prompt=True, tokenize=False)

def measure(model, name, batch_size=1, new_tokens=200):
    """Generate exactly new_tokens per prompt and report tokens/second."""
    inputs = tokenizer([prompt_text] * batch_size, return_tensors="pt",
                       padding=True, add_special_tokens=False).to(model.device)
    model.generate(**inputs, max_new_tokens=5, do_sample=False)   # warm-up, not timed
    torch.cuda.synchronize()
    start = time.time()
    model.generate(**inputs, max_new_tokens=new_tokens, min_new_tokens=new_tokens, do_sample=False)
    torch.cuda.synchronize()
    seconds = time.time() - start
    total = batch_size * new_tokens
    print(f"{name:28s} batch={batch_size:2d}  {seconds:6.1f} s  "
          f"{total / seconds:6.1f} tokens/s  ({seconds / batch_size:5.1f} s per prompt)")
    return seconds

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
print(f"4-bit model memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB\n")

measure(model_4bit, "A. 4-bit", batch_size=1)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

4-bit model memory: 1.07 GB

A. 4-bit                     batch= 1    12.8 s    15.6 tokens/s  ( 12.8 s per prompt)


12.841275453567505

In [3]:
import gc

# Free the 4-bit model first, so memory numbers are clean
del model_4bit
gc.collect()
torch.cuda.empty_cache()

model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
print(f"16-bit model memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB\n")

measure(model_fp16, "B. 16-bit", batch_size=1);

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

16-bit model memory: 2.88 GB

B. 16-bit                    batch= 1     7.7 s    26.1 tokens/s  (  7.7 s per prompt)


In [4]:
for bs in [1, 4, 8, 16]:
    measure(model_fp16, "C. 16-bit batched", batch_size=bs)
    torch.cuda.empty_cache()

print(f"\nPeak GPU memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

C. 16-bit batched            batch= 1    20.8 s     9.6 tokens/s  ( 20.8 s per prompt)
C. 16-bit batched            batch= 4    16.0 s    50.1 tokens/s  (  4.0 s per prompt)
C. 16-bit batched            batch= 8    12.7 s   125.7 tokens/s  (  1.6 s per prompt)
C. 16-bit batched            batch=16     9.0 s   355.5 tokens/s  (  0.6 s per prompt)

Peak GPU memory: 3.11 GB


In [5]:
import statistics

order = [16, 8, 4, 1, 1, 4, 8, 16, 16, 8, 4, 1]   # each size 3 times, mixed order
results = {1: [], 4: [], 8: [], 16: []}

for bs in order:
    seconds = measure(model_fp16, "repeat", batch_size=bs)
    results[bs].append(seconds)
    torch.cuda.empty_cache()

print("\n=== Median of 3 runs (200 tokens per prompt) ===")
for bs in [1, 4, 8, 16]:
    med = statistics.median(results[bs])
    spread = max(results[bs]) - min(results[bs])
    print(f"batch={bs:2d}  median {med:5.1f} s  |  {bs * 200 / med:6.1f} tokens/s  |  "
          f"{med / bs:4.2f} s per prompt  |  spread {spread:4.1f} s")

repeat                       batch=16    11.5 s   277.8 tokens/s  (  0.7 s per prompt)
repeat                       batch= 8     8.6 s   185.9 tokens/s  (  1.1 s per prompt)
repeat                       batch= 4    14.9 s    53.9 tokens/s  (  3.7 s per prompt)
repeat                       batch= 1     8.3 s    24.0 tokens/s  (  8.3 s per prompt)
repeat                       batch= 1     7.9 s    25.2 tokens/s  (  7.9 s per prompt)
repeat                       batch= 4     8.2 s    97.9 tokens/s  (  2.0 s per prompt)
repeat                       batch= 8     8.7 s   184.8 tokens/s  (  1.1 s per prompt)
repeat                       batch=16     8.9 s   358.2 tokens/s  (  0.6 s per prompt)
repeat                       batch=16     8.5 s   376.3 tokens/s  (  0.5 s per prompt)
repeat                       batch= 8     8.6 s   186.4 tokens/s  (  1.1 s per prompt)
repeat                       batch= 4     8.5 s    93.6 tokens/s  (  2.1 s per prompt)
repeat                       batch= 1     8